# Categorical Encoding: Ordinal & Integer Encoding

**Ordinal encoding** consists of replacing categorical values with integers. 

However, in machine learning, there is a fundamental distinction between:
1. **True Ordinal Encoding**: Assigning integers that reflect a **natural, meaningful order** (e.g., `Poor = 1`, `Fair = 2`, `Good = 3`, `Excellent = 4`).
2. **Arbitrary Integer Encoding**: Assigning arbitrary integers ($0, 1, 2, \dots$) to **nominal** (unordered) categories.

---

## Learning Objectives
By the end of this notebook, you will be able to:
1. Distinguish between **truly ordinal** variables and **nominal** variables.
2. Understand which machine learning algorithms are distorted by arbitrary integer encoding (linear models, distance-based models) and which can utilize them (decision trees, gradient boosting).
3. Specify explicit, domain-accurate category orderings using **`OrdinalEncoder(categories=[...])`** and **pandas `.map()`**.
4. Handle unseen test categories safely using `handle_unknown='use_encoded_value'`.
5. Integrate ordinal encoding into a **`ColumnTransformer`** workflow.

---
## 1. Nominal vs. Ordinal Categories: The Algorithm Impact

| Category Type | Definition | Example | Best Encoding Method |
| :--- | :--- | :--- | :--- |
| **Ordinal** | Has an intrinsic, meaningful ranking | `Poor < Fair < Average < Good < Excellent` | **True Ordinal Encoding** (explicit integer rank mapping) |
| **Nominal** | No intrinsic ranking | `Neighborhood` (CollgCr, Veenker, Crawfor) | **One-Hot Encoding** (linear models) or **Frequency/Target Encoding** (high cardinality) |

### ⚠️ Why Arbitrary Integer Encoding Distorts Linear & Distance Models
If we assign integers arbitrarily to nominal features:
$$\text{CollgCr} = 0, \quad \text{Veenker} = 1, \quad \text{Crawfor} = 2$$
A linear model calculates:
$$y = \beta \cdot \text{Neighborhood}$$
This forces the model to assume that $\text{Crawfor} = 2 \times \text{Veenker}$, and that $\text{Crawfor} > \text{Veenker} > \text{CollgCr}$! This artificial ordering severely distorts linear regressions, logistic regressions, SVMs, and neural networks.

**Exception: Tree-Based Models** (Decision Trees, Random Forests, LightGBM, CatBoost) make non-linear threshold cuts (`Feature == Category` or `Feature <= k`), so they can handle arbitrary integer-encoded nominal variables effectively.

In [1]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder
from sklearn.compose import ColumnTransformer

pd.set_option('display.max_columns', None)

---
## 2. Loading Data & Train/Test Split

We use features from the Ames House Prices dataset:
- **True Ordinal Features**:
  - `ExterQual`: Exterior material quality (`Po` < `Fa` < `TA` < `Gd` < `Ex`)
  - `KitchenQual`: Kitchen quality (`Po` < `Fa` < `TA` < `Gd` < `Ex`)
  - `BsmtQual`: Height of the basement (`Po` < `Fa` < `TA` < `Gd` < `Ex`)
- **Nominal Feature**:
  - `Neighborhood`: Physical location within Ames city limits (unordered)

In [2]:
cols_to_use = [
    "Neighborhood",
    "ExterQual",
    "KitchenQual",
    "BsmtQual",
    "SalePrice",
]

data_path = '../datasets/houseprice.csv'
if not os.path.exists(data_path):
    data_path = 'https://raw.githubusercontent.com/cmps460s26/cmps460-content/refs/heads/main/examples/03.dp/datasets/houseprice.csv'

data = pd.read_csv(data_path, usecols=cols_to_use)

# Fill missing basement quality with 'None' (representing no basement)
data['BsmtQual'] = data['BsmtQual'].fillna('None')

# Separate predictors and target
X = data.drop("SalePrice", axis=1)
y = data["SalePrice"]

# Train/Test Split BEFORE learning category mappings
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=0
)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape:  {X_test.shape}")
X_train.head()

X_train shape: (1022, 4)
X_test shape:  (438, 4)


,Neighborhood,ExterQual,BsmtQual,KitchenQual
64,CollgCr,TA,Gd,Gd
682,ClearCr,TA,Gd,Gd
960,BrkSide,TA,TA,TA
1384,Edwards,TA,TA,TA
1100,SWISU,TA,TA,Fa


---
## 3. Method 1: Explicit Ordinal Encoding with Pandas `.map()`

When we have domain knowledge of the category hierarchy, pandas `.map()` provides the most transparent, readable implementation.

In [3]:
# Define explicit quality order (higher integer = better quality)
quality_mapping = {
    'None': 0,   # No feature present
    'Po': 1,     # Poor
    'Fa': 2,     # Fair
    'TA': 3,     # Typical/Average
    'Gd': 4,     # Good
    'Ex': 5      # Excellent
}

print("Ordinal Quality Hierarchy:")
for cat, score in quality_mapping.items():
    print(f"  {cat:6s} -> {score}")

Ordinal Quality Hierarchy:
  None   -> 0
  Po     -> 1
  Fa     -> 2
  TA     -> 3
  Gd     -> 4
  Ex     -> 5


In [4]:
# Create copies for pandas demonstration
X_train_pd = X_train.copy()
X_test_pd = X_test.copy()

# Map ordinal columns
for col in ['ExterQual', 'KitchenQual', 'BsmtQual']:
    X_train_pd[f"{col}_encoded"] = X_train_pd[col].map(quality_mapping)
    X_test_pd[f"{col}_encoded"]  = X_test_pd[col].map(quality_mapping)

# Inspect original vs encoded columns
sample_cols = ['ExterQual', 'ExterQual_encoded', 'KitchenQual', 'KitchenQual_encoded']
X_train_pd[sample_cols].head(6)

,ExterQual,ExterQual_encoded,KitchenQual,KitchenQual_encoded
64,TA,3,Gd,4
682,TA,3,Gd,4
960,TA,3,TA,3
1384,TA,3,TA,3
1100,TA,3,Fa,2
416,TA,3,TA,3


---
## 4. Method 2: Scikit-Learn `OrdinalEncoder` with Explicit Categories

To preserve an explicit order within scikit-learn pipelines, we pass an ordered list of lists to the `categories` parameter of `OrdinalEncoder`.

We also set `handle_unknown='use_encoded_value', unknown_value=-1` to ensure that any unexpected category encountered in test/production data is safely flagged without raising an exception.

In [5]:
# Ordered lists for each feature
ordinal_order = ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex']
ordinal_cols = ['ExterQual', 'KitchenQual', 'BsmtQual']

# Configure OrdinalEncoder with explicit hierarchy
explicit_encoder = OrdinalEncoder(
    categories=[
        ['Po', 'Fa', 'TA', 'Gd', 'Ex'],         # ExterQual has no 'None'
        ['Po', 'Fa', 'TA', 'Gd', 'Ex'],         # KitchenQual has no 'None'
        ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex']   # BsmtQual includes 'None'
    ],
    handle_unknown='use_encoded_value',
    unknown_value=-1
)
explicit_encoder.set_output(transform="pandas")

# Fit on training set
explicit_encoder.fit(X_train[ordinal_cols])

# Transform train and test
X_train_ord = explicit_encoder.transform(X_train[ordinal_cols])
X_test_ord  = explicit_encoder.transform(X_test[ordinal_cols])

X_train_ord.head()

,ExterQual,KitchenQual,BsmtQual
64,2.0,3.0,4.0
682,2.0,3.0,4.0
960,2.0,2.0,3.0
1384,2.0,2.0,3.0
1100,2.0,1.0,3.0


---
## 5. Arbitrary Integer Encoding for Nominal Features (Tree Models)

If you are feeding high-cardinality nominal features (like `Neighborhood`) into a **decision tree, Random Forest, or XGBoost/LightGBM model**, you can let `OrdinalEncoder` assign arbitrary integer codes.

In [6]:
# Default OrdinalEncoder (orders categories alphabetically or by appearance)
nominal_encoder = OrdinalEncoder(
    handle_unknown='use_encoded_value',
    unknown_value=-1
)
nominal_encoder.set_output(transform="pandas")

# Fit and transform Neighborhood
nominal_encoder.fit(X_train[['Neighborhood']])
X_train_neigh = nominal_encoder.transform(X_train[['Neighborhood']])

print("Sample of arbitrary integer encoding on nominal 'Neighborhood':")
sample_neigh = pd.concat([X_train[['Neighborhood']], X_train_neigh.rename(columns={'Neighborhood': 'Neighborhood_Code'})], axis=1)
sample_neigh.head(6)

Sample of arbitrary integer encoding on nominal 'Neighborhood':


,Neighborhood,Neighborhood_Code
64,CollgCr,5.0
682,ClearCr,4.0
960,BrkSide,3.0
1384,Edwards,7.0
1100,SWISU,18.0
416,Sawyer,19.0


---
## 6. Production Workflow: `ColumnTransformer`

In practice, we combine both approaches into a unified `ColumnTransformer`:
- **Explicit Ordinal Encoding** for quality features.
- **Arbitrary Integer Encoding** (or One-Hot Encoding) for nominal features.

In [7]:
# Build unified ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('ordinal_qual', explicit_encoder, ordinal_cols),
        ('nominal_neigh', nominal_encoder, ['Neighborhood'])
    ],
    remainder='passthrough'
)
preprocessor.set_output(transform="pandas")

# Fit on training data and transform both sets
X_train_final = preprocessor.fit_transform(X_train)
X_test_final  = preprocessor.transform(X_test)

print(f"Final transformed feature matrix shape: {X_train_final.shape}")
X_train_final.head()

Final transformed feature matrix shape: (1022, 4)


,ordinal_qual__ExterQual,ordinal_qual__KitchenQual,ordinal_qual__BsmtQual,nominal_neigh__Neighborhood
64,2.0,3.0,4.0,5.0
682,2.0,3.0,4.0,4.0
960,2.0,2.0,3.0,3.0
1384,2.0,2.0,3.0,7.0
1100,2.0,1.0,3.0,18.0


---
## 7. Summary & Best Practices Checklist

| Feature Type | Recommended Method | Note |
| :--- | :--- | :--- |
| **True Ordinal** | `OrdinalEncoder(categories=[ordered_lists])` or `.map(order_dict)` | Reflects true domain hierarchy; safe for all model types (linear, SVM, trees). |
| **Nominal (Tree Models)** | `OrdinalEncoder()` with arbitrary integers | Compact; trees handle non-linear integer codes well. |
| **Nominal (Linear Models)** | **One-Hot Encoding** (Notebook 01) | Never use arbitrary integer encoding with linear models; it creates false mathematical ordering. |
| **Unseen Categories** | `handle_unknown='use_encoded_value', unknown_value=-1` | Prevents runtime crashes on unseen categories in production. |
